# 5. Payer & Utilization Intelligence
## Insurance Plan Performance and Member Attribution Analytics

Strategy: LIMIT 1000 on BigQuery reads -> Local CSV -> Pandas -> Spark

## Available Tables:
- **OMOP**: 24 tables
- **Medicare**: 6 tables
- **Dual Enrollment**: 1 table (SDOH)
- **CMS Codes**: 3 tables

## Pipeline Architecture:
- **Bronze**: 10 tables (payer, utilization, cost data)
- **Silver**: 6 intermediate layers (plan aggregations)
- **Gold**: 15 vertical layers -> 4 final metrics

## Final Metrics:
1. **Risk-Adjusted Utilization Rate** - HCC-normalized service usage
2. **Benefit Design Effectiveness** - Coverage optimization score
3. **High-Value Care Penetration** - Quality service adoption rate
4. **Member Attribution Stability** - Continuity of coverage index

In [1]:
from google.cloud import bigquery
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import *
import os
from datetime import datetime
import networkx as nx
import json

In [2]:
print("Initializing Spark...")
spark = SparkSession.builder \
    .appName("PayerUtilizationIntelligence") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Initializing Spark...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/02 23:02:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1
Spark UI: http://mac:4045


25/12/02 23:02:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/02 23:02:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/12/02 23:02:31 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/12/02 23:02:31 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/12/02 23:02:31 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [3]:
# Configuration
GCP_PROJECT = "opportune-ruler-447319-b3"
BQ_CLIENT = bigquery.Client(project=GCP_PROJECT)

# Local data directory
LOCAL_DATA_DIR = "./5_data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# BigQuery datasets
DATASETS = {
    'omop': 'bigquery-public-data.cms_synthetic_patient_data_omop',
    'medicare': 'bigquery-public-data.cms_medicare',
    'dual_enrollment': 'bigquery-public-data.sdoh_cms_dual_eligible_enrollment',
    'cms_codes': 'bigquery-public-data.cms_codes'
}

print(f"\nConfiguration:")
print(f"  GCP Project: {GCP_PROJECT}")
print(f"  Local Data: {LOCAL_DATA_DIR}")
print(f"  Datasets: {len(DATASETS)}")


Configuration:
  GCP Project: opportune-ruler-447319-b3
  Local Data: ./5_data
  Datasets: 4


# STEP 1: Download Limited Data from BigQuery (LIMIT 1000)

In [40]:
def download_table_limited(dataset_name, table_name, limit=1000):
    """
    Download limited rows from BigQuery table to local CSV
    """
    full_table = f"{DATASETS[dataset_name]}.{table_name}"
    csv_path = f"{LOCAL_DATA_DIR}/{table_name}.csv"
    
    if os.path.exists(csv_path):
        print(f"  Skip {table_name} (already exists)")
        return csv_path
    
    try:
        query = f"SELECT * FROM `{full_table}` LIMIT {limit}"
        df_pd = BQ_CLIENT.query(query).to_dataframe()
        df_pd.to_csv(csv_path, index=False)
        print(f"  Downloaded {table_name}: {len(df_pd)} rows -> {csv_path}")
        return csv_path
    except Exception as e:
        print(f"  Error downloading {table_name}: {e}")
        return None

In [41]:
print("\n" + "="*80)
print("DOWNLOADING TABLES (LIMIT 1000 each)")
print("="*80)

tables_to_download = [
    ('omop', 'person'),
    ('omop', 'observation_period'),
    ('omop', 'payer_plan_period'),
    ('omop', 'condition_occurrence'),
    ('omop', 'procedure_occurrence'),
    ('omop', 'drug_exposure'),
    ('omop', 'cost'),
    ('omop', 'care_site'),
    ('omop', 'provider'),
    ('medicare', 'inpatient_charges_2011')
]

downloaded_files = {}
for dataset, table in tables_to_download:
    path = download_table_limited(dataset, table, limit=1000)
    if path:
        downloaded_files[table] = path

print(f"\nDownloaded {len(downloaded_files)} tables")


DOWNLOADING TABLES (LIMIT 1000 each)
  Skip person (already exists)
  Skip observation_period (already exists)
  Skip payer_plan_period (already exists)
  Skip condition_occurrence (already exists)
  Skip procedure_occurrence (already exists)
  Skip drug_exposure (already exists)
  Skip cost (already exists)
  Skip care_site (already exists)
  Skip provider (already exists)
  Skip inpatient_charges_2011 (already exists)

Downloaded 10 tables


# STEP 2: Load CSV to Pandas then Spark (Bronze Layer)

In [4]:
print("\n" + "="*80)
print("BRONZE LAYER: Load CSV -> Pandas -> Spark")
print("="*80)

def load_csv_to_spark(table_name):
    csv_path = f"{LOCAL_DATA_DIR}/{table_name}.csv"
    if not os.path.exists(csv_path):
        print(f"  Warning: {table_name}.csv not found")
        return None
    
    try:
        df_pd = pd.read_csv(csv_path)
        df_spark = spark.createDataFrame(df_pd)
        print(f"  Loaded {table_name}: {df_spark.count()} rows, {len(df_spark.columns)} cols")
        return df_spark
    except Exception as e:
        print(f"  Error loading {table_name}: {e}")
        return None

bronze_person = load_csv_to_spark('person')
bronze_obs_period = load_csv_to_spark('observation_period')
bronze_payer = load_csv_to_spark('payer_plan_period')
bronze_condition = load_csv_to_spark('condition_occurrence')
bronze_procedure = load_csv_to_spark('procedure_occurrence')
bronze_drug = load_csv_to_spark('drug_exposure')
bronze_cost = load_csv_to_spark('cost')
bronze_care_site = load_csv_to_spark('care_site')
bronze_provider = load_csv_to_spark('provider')
bronze_inpatient = load_csv_to_spark('inpatient_charges_2011')

print("\nBronze layer loaded successfully")


BRONZE LAYER: Load CSV -> Pandas -> Spark


  Loaded person: 1000 rows, 18 cols
  Loaded observation_period: 1000 rows, 5 cols
  Loaded payer_plan_period: 1000 rows, 17 cols
  Loaded condition_occurrence: 1000 rows, 16 cols
  Loaded procedure_occurrence: 1000 rows, 14 cols
  Loaded drug_exposure: 1000 rows, 23 cols
  Loaded cost: 1000 rows, 22 cols
  Loaded care_site: 1000 rows, 6 cols
  Loaded provider: 1000 rows, 13 cols
  Loaded inpatient_charges_2011: 1000 rows, 12 cols

Bronze layer loaded successfully


# STEP 3: SILVER LAYER (6 Intermediate Transformations)

In [5]:
print("\n" + "="*80)
print("SILVER 1: Member Demographics & Enrollment")
print("="*80)

silver_member_enrollment = bronze_person \
    .join(bronze_obs_period, 'person_id', 'left') \
    .join(bronze_payer, 'person_id', 'left') \
    .select(
        F.col('person_id'),
        F.col('gender_concept_id'),
        F.col('birth_datetime'),
        F.col('race_concept_id'),
        F.col('ethnicity_concept_id'),
        F.col('observation_period_start_date'),
        F.col('observation_period_end_date'),
        F.col('payer_plan_period_start_date'),
        F.col('payer_plan_period_end_date'),
        F.coalesce(F.col('payer_concept_id'), F.lit(0)).alias('payer_concept_id'),
        F.coalesce(F.col('plan_concept_id'), F.lit(0)).alias('plan_concept_id')
    ) \
    .withColumn(
        'birth_datetime_converted',
        F.to_date(F.col('birth_datetime').cast('string'))
    ) \
    .withColumn(
        'payer_plan_period_start_date_converted',
        F.to_date(F.col('payer_plan_period_start_date').cast('string'))
    ) \
    .withColumn(
        'payer_plan_period_end_date_converted',
        F.to_date(F.col('payer_plan_period_end_date').cast('string'))
    ) \
    .withColumn(
        'age',
        F.floor(F.datediff(F.current_date(), F.col('birth_datetime_converted')) / 365.25)
    ) \
    .withColumn(
        'enrollment_days',
        F.datediff(
            F.coalesce(F.col('payer_plan_period_end_date_converted'), F.current_date()),
            F.col('payer_plan_period_start_date_converted')
        )
    ) \
    .drop('birth_datetime', 'payer_plan_period_start_date', 'payer_plan_period_end_date') \
    .withColumnRenamed('birth_datetime_converted', 'birth_datetime') \
    .withColumnRenamed('payer_plan_period_start_date_converted', 'payer_plan_period_start_date') \
    .withColumnRenamed('payer_plan_period_end_date_converted', 'payer_plan_period_end_date')

print(f"Silver 1 complete: {silver_member_enrollment.count()} members")


SILVER 1: Member Demographics & Enrollment
Silver 1 complete: 1000 members


In [6]:
print("\n" + "="*80)
print("SILVER 2: Service Utilization by Member")
print("="*80)

condition_util = bronze_condition.groupBy('person_id').agg(
    F.count('*').alias('condition_count'),
    F.countDistinct('condition_concept_id').alias('unique_conditions')
)

procedure_util = bronze_procedure.groupBy('person_id').agg(
    F.count('*').alias('procedure_count'),
    F.countDistinct('procedure_concept_id').alias('unique_procedures')
)

drug_util = bronze_drug.groupBy('person_id').agg(
    F.count('*').alias('drug_count'),
    F.countDistinct('drug_concept_id').alias('unique_drugs'),
    F.sum('days_supply').alias('total_days_supply')
)

silver_utilization = silver_member_enrollment.select('person_id') \
    .join(condition_util, 'person_id', 'left') \
    .join(procedure_util, 'person_id', 'left') \
    .join(drug_util, 'person_id', 'left') \
    .fillna(0)

print(f"Silver 2 complete: {silver_utilization.count()} records")


SILVER 2: Service Utilization by Member
Silver 2 complete: 1000 records


In [7]:
print("\n" + "="*80)
print("SILVER 3: Cost Attribution by Payer")
print("="*80)

payer_mapping = bronze_payer.select(
    F.col('payer_plan_period_id').alias('payer_period_id'),
    'person_id', 
    'payer_concept_id'
)

silver_cost_by_payer = bronze_cost \
    .join(payer_mapping, 
          bronze_cost.payer_plan_period_id == payer_mapping.payer_period_id, 
          'left') \
    .groupBy('person_id', 'payer_concept_id').agg(
        F.sum('total_paid').alias('total_paid_by_payer'),
        F.sum('paid_by_payer').alias('payer_paid'),
        F.sum('paid_by_patient').alias('patient_paid'),
        F.count('*').alias('cost_event_count')
    )

print(f"Silver 3 complete: {silver_cost_by_payer.count()} payer-member combinations")


SILVER 3: Cost Attribution by Payer
Silver 3 complete: 1 payer-member combinations


In [8]:
print("\n" + "="*80)
print("SILVER 4: Provider Network Attribution")
print("="*80)

member_provider = bronze_condition \
    .join(bronze_provider, bronze_condition.provider_id == bronze_provider.provider_id, 'left') \
    .groupBy('person_id', bronze_provider.provider_id).agg(
        F.count('*').alias('visit_count')
    )

w = W.partitionBy('person_id').orderBy(F.desc('visit_count'))
silver_provider_attribution = member_provider \
    .withColumn('rank', F.row_number().over(w)) \
    .filter(F.col('rank') == 1) \
    .select('person_id', F.col('provider_id').alias('attributed_provider_id'), 'visit_count')

print(f"Silver 4 complete: {silver_provider_attribution.count()} attributions")


SILVER 4: Provider Network Attribution
Silver 4 complete: 1000 attributions


In [9]:
print("\n" + "="*80)
print("SILVER 5: Risk Score Calculation (HCC-like)")
print("="*80)

silver_risk_scores = silver_member_enrollment \
    .join(silver_utilization.select('person_id', 'unique_conditions'), 'person_id', 'left') \
    .withColumn(
        'age_risk_factor',
        F.when(F.col('age') >= 85, 2.5)
         .when(F.col('age') >= 75, 2.0)
         .when(F.col('age') >= 65, 1.5)
         .when(F.col('age') >= 50, 1.2)
         .otherwise(1.0)
    ) \
    .withColumn(
        'condition_risk_factor',
        F.when(F.col('unique_conditions') >= 10, 2.0)
         .when(F.col('unique_conditions') >= 5, 1.5)
         .when(F.col('unique_conditions') >= 2, 1.2)
         .otherwise(1.0)
    ) \
    .withColumn(
        'hcc_risk_score',
        F.col('age_risk_factor') * F.col('condition_risk_factor')
    ) \
    .select('person_id', 'hcc_risk_score', 'age_risk_factor', 'condition_risk_factor')

print(f"Silver 5 complete: {silver_risk_scores.count()} risk scores")


SILVER 5: Risk Score Calculation (HCC-like)
Silver 5 complete: 1000 risk scores


In [10]:
print("\n" + "="*80)
print("SILVER 6: Plan Benefit Design Features")
print("="*80)

plan_agg = bronze_payer.groupBy('payer_concept_id', 'plan_concept_id').agg(
    F.count('person_id').alias('member_count'),
    F.countDistinct('person_id').alias('unique_members')
)

cost_agg = silver_cost_by_payer.groupBy('payer_concept_id').agg(
    F.avg('total_paid_by_payer').alias('avg_total_cost'),
    F.avg('payer_paid').alias('avg_payer_share'),
    F.avg('patient_paid').alias('avg_patient_share')
)

silver_plan_features = plan_agg \
    .join(cost_agg, 'payer_concept_id', 'left') \
    .withColumn(
        'payer_cost_share_pct',
        F.col('avg_payer_share') / F.greatest(F.col('avg_total_cost'), F.lit(1)) * 100
    )

print(f"Silver 6 complete: {silver_plan_features.count()} plan features")


SILVER 6: Plan Benefit Design Features
Silver 6 complete: 1 plan features


# STEP 4: GOLD LAYER (15 Vertical Transformations)

In [11]:
print("\n" + "="*80)
print("GOLD LEVEL 1: Integrated Member Profile")
print("="*80)

# Clean up each source first
member_base = silver_member_enrollment.select(
    'person_id',
    'gender_concept_id',
    'birth_datetime',
    'race_concept_id',
    'ethnicity_concept_id',
    'observation_period_start_date',
    'observation_period_end_date',
    'payer_plan_period_start_date',
    'payer_plan_period_end_date',
    'payer_concept_id',
    'plan_concept_id',
    'age',
    'enrollment_days'
)

gold_l1_member_profile = member_base \
    .join(silver_utilization, 'person_id', 'left') \
    .join(
        silver_cost_by_payer.select(
            F.col('person_id').alias('cost_pid'),
            F.col('payer_concept_id').alias('cost_payer_id'),
            'total_paid_by_payer',
            'payer_paid',
            'patient_paid',
            'cost_event_count'
        ),
        member_base.person_id == F.col('cost_pid'),
        'left'
    ) \
    .drop('cost_pid') \
    .withColumn(
        'payer_concept_id_final',
        F.coalesce(F.col('payer_concept_id'), F.col('cost_payer_id'))
    ) \
    .drop('payer_concept_id', 'cost_payer_id') \
    .withColumnRenamed('payer_concept_id_final', 'payer_concept_id') \
    .join(silver_provider_attribution, 'person_id', 'left') \
    .join(silver_risk_scores, 'person_id', 'left') \
    .fillna(0)

print(f"Gold L1 complete: {gold_l1_member_profile.count()} integrated profiles")


GOLD LEVEL 1: Integrated Member Profile


Gold L1 complete: 1000 integrated profiles


In [12]:
print("\n" + "="*80)
print("GOLD LEVEL 2: Utilization Density Metrics")
print("="*80)

gold_l2_utilization_density = gold_l1_member_profile \
    .withColumn(
        'enrollment_years',
        F.greatest(F.col('enrollment_days') / 365.25, F.lit(0.5))
    ) \
    .withColumn(
        'annual_condition_rate',
        F.col('condition_count') / F.col('enrollment_years')
    ) \
    .withColumn(
        'annual_procedure_rate',
        F.col('procedure_count') / F.col('enrollment_years')
    ) \
    .withColumn(
        'annual_drug_rate',
        F.col('drug_count') / F.col('enrollment_years')
    ) \
    .withColumn(
        'total_annual_utilization',
        F.col('annual_condition_rate') + F.col('annual_procedure_rate') + F.col('annual_drug_rate')
    )

print("Gold L2 complete")


GOLD LEVEL 2: Utilization Density Metrics
Gold L2 complete


In [13]:
print("\n" + "="*80)
print("GOLD LEVEL 3: Risk-Adjusted Utilization")
print("="*80)

gold_l3_risk_adjusted = gold_l2_utilization_density \
    .withColumn(
        'expected_utilization',
        F.col('hcc_risk_score') * 10.0
    ) \
    .withColumn(
        'utilization_vs_expected',
        F.col('total_annual_utilization') / F.greatest(F.col('expected_utilization'), F.lit(1))
    ) \
    .withColumn(
        'risk_adjusted_utilization_rate',
        F.col('total_annual_utilization') / F.greatest(F.col('hcc_risk_score'), F.lit(1))
    )

print("Gold L3 complete")


GOLD LEVEL 3: Risk-Adjusted Utilization
Gold L3 complete


In [14]:
print("\n" + "="*80)
print("GOLD LEVEL 4: Cost per Member per Month (PMPM)")
print("="*80)

gold_l4_pmpm = gold_l3_risk_adjusted \
    .withColumn(
        'enrollment_months',
        F.greatest(F.col('enrollment_days') / 30.0, F.lit(1))
    ) \
    .withColumn(
        'total_cost_pmpm',
        F.col('total_paid_by_payer') / F.col('enrollment_months')
    ) \
    .withColumn(
        'payer_cost_pmpm',
        F.col('payer_paid') / F.col('enrollment_months')
    ) \
    .withColumn(
        'patient_cost_pmpm',
        F.col('patient_paid') / F.col('enrollment_months')
    )

print("Gold L4 complete")


GOLD LEVEL 4: Cost per Member per Month (PMPM)
Gold L4 complete


In [15]:
print("\n" + "="*80)
print("GOLD LEVEL 5: Risk-Adjusted PMPM")
print("="*80)

gold_l5_risk_adjusted_pmpm = gold_l4_pmpm \
    .withColumn(
        'expected_pmpm',
        F.col('hcc_risk_score') * 500.0
    ) \
    .withColumn(
        'pmpm_efficiency_ratio',
        F.col('total_cost_pmpm') / F.greatest(F.col('expected_pmpm'), F.lit(100))
    ) \
    .withColumn(
        'risk_adjusted_pmpm',
        F.col('total_cost_pmpm') / F.greatest(F.col('hcc_risk_score'), F.lit(1))
    )

print("Gold L5 complete")


GOLD LEVEL 5: Risk-Adjusted PMPM
Gold L5 complete


In [16]:
print("\n" + "="*80)
print("GOLD LEVEL 6: Plan Coverage Effectiveness")
print("="*80)

plan_cost_share = silver_plan_features.select(
    F.col('payer_concept_id').alias('plan_payer_id'),
    'payer_cost_share_pct'
)

gold_l6_coverage = gold_l5_risk_adjusted_pmpm \
    .join(plan_cost_share, 
          gold_l5_risk_adjusted_pmpm.payer_concept_id == plan_cost_share.plan_payer_id, 
          'left') \
    .drop('plan_payer_id') \
    .withColumn(
        'patient_burden_ratio',
        F.col('patient_paid') / F.greatest(F.col('total_paid_by_payer'), F.lit(1))
    ) \
    .withColumn(
        'coverage_adequacy_score',
        100 * (1 - F.least(F.col('patient_burden_ratio'), F.lit(1)))
    )

print("Gold L6 complete")


GOLD LEVEL 6: Plan Coverage Effectiveness
Gold L6 complete


In [17]:
print("\n" + "="*80)
print("GOLD LEVEL 7: High-Value Service Utilization")
print("="*80)

gold_l7_high_value = gold_l6_coverage \
    .withColumn(
        'preventive_procedure_ratio',
        F.when(F.col('procedure_count') > 0, 
               F.lit(0.3) * F.col('procedure_count') / F.col('procedure_count'))
         .otherwise(0)
    ) \
    .withColumn(
        'chronic_drug_adherence_proxy',
        F.when(F.col('unique_conditions') >= 2,
               F.least(F.col('total_days_supply') / (F.col('enrollment_days') * F.col('unique_conditions')), F.lit(1)))
         .otherwise(0)
    ) \
    .withColumn(
        'high_value_care_score',
        (F.col('preventive_procedure_ratio') * 50) + (F.col('chronic_drug_adherence_proxy') * 50)
    )

print("Gold L7 complete")


GOLD LEVEL 7: High-Value Service Utilization
Gold L7 complete


In [18]:
print("\n" + "="*80)
print("GOLD LEVEL 8: Provider Attribution Continuity")
print("="*80)

gold_l8_attribution = gold_l7_high_value \
    .withColumn(
        'has_attributed_provider',
        F.when(F.col('attributed_provider_id').isNotNull(), 1).otherwise(0)
    ) \
    .withColumn(
        'attribution_strength',
        F.when(F.col('visit_count') >= 5, 1.0)
         .when(F.col('visit_count') >= 3, 0.7)
         .when(F.col('visit_count') >= 1, 0.4)
         .otherwise(0)
    ) \
    .withColumn(
        'continuity_score',
        F.col('has_attributed_provider') * F.col('attribution_strength') * 100
    )

print("Gold L8 complete")


GOLD LEVEL 8: Provider Attribution Continuity
Gold L8 complete


In [19]:
print("\n" + "="*80)
print("GOLD LEVEL 9: Member Tenure & Stability")
print("="*80)

gold_l9_stability = gold_l8_attribution \
    .withColumn(
        'tenure_years',
        F.col('enrollment_days') / 365.25
    ) \
    .withColumn(
        'tenure_stability_factor',
        F.when(F.col('tenure_years') >= 5, 1.0)
         .when(F.col('tenure_years') >= 3, 0.8)
         .when(F.col('tenure_years') >= 1, 0.6)
         .otherwise(0.3)
    ) \
    .withColumn(
        'enrollment_gap_risk',
        F.when(F.col('payer_plan_period_end_date').isNull(), 0)
         .otherwise(1)
    )

print("Gold L9 complete")


GOLD LEVEL 9: Member Tenure & Stability
Gold L9 complete


In [20]:
print("\n" + "="*80)
print("GOLD LEVEL 10: Plan Performance Benchmarking")
print("="*80)

plan_benchmarks = gold_l9_stability \
    .groupBy('payer_concept_id').agg(
        F.mean('total_cost_pmpm').alias('plan_avg_pmpm'),
        F.mean('risk_adjusted_pmpm').alias('plan_avg_risk_adj_pmpm'),
        F.mean('coverage_adequacy_score').alias('plan_avg_coverage'),
        F.count('*').alias('plan_member_count')
    ).select(
        F.col('payer_concept_id').alias('benchmark_payer_id'),
        'plan_avg_pmpm',
        'plan_avg_risk_adj_pmpm',
        'plan_avg_coverage',
        'plan_member_count'
    )

gold_l10_benchmarked = gold_l9_stability \
    .join(plan_benchmarks, 
          gold_l9_stability.payer_concept_id == plan_benchmarks.benchmark_payer_id, 
          'left') \
    .drop('benchmark_payer_id') \
    .withColumn(
        'pmpm_vs_plan_avg',
        F.col('total_cost_pmpm') / F.greatest(F.col('plan_avg_pmpm'), F.lit(100))
    ) \
    .withColumn(
        'coverage_vs_plan_avg',
        F.col('coverage_adequacy_score') / F.greatest(F.col('plan_avg_coverage'), F.lit(50))
    )

print("Gold L10 complete")


GOLD LEVEL 10: Plan Performance Benchmarking
Gold L10 complete


In [21]:
print("\n" + "="*80)
print("GOLD LEVEL 11: Value-Based Care Alignment")
print("="*80)

gold_l11_vbc = gold_l10_benchmarked \
    .withColumn(
        'quality_utilization_ratio',
        F.col('high_value_care_score') / F.greatest(F.col('total_annual_utilization'), F.lit(1))
    ) \
    .withColumn(
        'cost_quality_efficiency',
        F.col('high_value_care_score') / F.greatest(F.col('risk_adjusted_pmpm'), F.lit(100))
    ) \
    .withColumn(
        'vbc_alignment_score',
        (F.col('quality_utilization_ratio') * 0.6 + F.col('cost_quality_efficiency') * 0.4) * 100
    )

print("Gold L11 complete")


GOLD LEVEL 11: Value-Based Care Alignment
Gold L11 complete


In [22]:
print("\n" + "="*80)
print("GOLD LEVEL 12: Member Engagement Index")
print("="*80)

gold_l12_engagement = gold_l11_vbc \
    .withColumn(
        'utilization_engagement',
        F.when(F.col('total_annual_utilization') > 0, 1).otherwise(0)
    ) \
    .withColumn(
        'provider_engagement',
        F.col('has_attributed_provider')
    ) \
    .withColumn(
        'preventive_engagement',
        F.when(F.col('preventive_procedure_ratio') > 0.2, 1).otherwise(0)
    ) \
    .withColumn(
        'member_engagement_index',
        (F.col('utilization_engagement') + F.col('provider_engagement') + F.col('preventive_engagement')) / 3.0 * 100
    )

print("Gold L12 complete")


GOLD LEVEL 12: Member Engagement Index
Gold L12 complete


In [23]:
print("\n" + "="*80)
print("GOLD LEVEL 13: METRIC 1 - Risk-Adjusted Utilization Rate")
print("="*80)

gold_l13_metric1 = gold_l12_engagement \
    .withColumn(
        'normalized_utilization',
        F.col('total_annual_utilization') / 50.0
    ) \
    .withColumn(
        'normalized_risk',
        F.col('hcc_risk_score') / 2.5
    ) \
    .withColumn(
        'risk_adjusted_utilization_rate_final',
        F.least(
            (F.col('normalized_utilization') / F.greatest(F.col('normalized_risk'), F.lit(0.5))) * 100,
            F.lit(200)
        )
    )

print("Gold L13 complete: METRIC 1 calculated")


GOLD LEVEL 13: METRIC 1 - Risk-Adjusted Utilization Rate
Gold L13 complete: METRIC 1 calculated


In [24]:
print("\n" + "="*80)
print("GOLD LEVEL 14: METRIC 2 - Benefit Design Effectiveness")
print("="*80)

gold_l14_metric2 = gold_l13_metric1 \
    .withColumn(
        'benefit_design_effectiveness',
        (
            F.col('coverage_adequacy_score') * 0.4 +
            F.col('high_value_care_score') * 0.3 +
            (100 - F.least(F.col('patient_burden_ratio') * 100, F.lit(100))) * 0.3
        )
    )

print("Gold L14 complete: METRIC 2 calculated")


GOLD LEVEL 14: METRIC 2 - Benefit Design Effectiveness
Gold L14 complete: METRIC 2 calculated


In [25]:
print("\n" + "="*80)
print("GOLD LEVEL 15: METRICS 3 & 4")
print("="*80)

gold_l15_final_metrics = gold_l14_metric2 \
    .withColumn(
        'high_value_care_penetration',
        F.col('high_value_care_score')
    ) \
    .withColumn(
        'member_attribution_stability',
        (
            F.col('continuity_score') * 0.4 +
            F.col('tenure_stability_factor') * 30 +
            (1 - F.col('enrollment_gap_risk')) * 30
        )
    )

print("Gold L15 complete: All 4 metrics calculated")
print("\nFinal Metrics:")
print("  1. risk_adjusted_utilization_rate_final")
print("  2. benefit_design_effectiveness")
print("  3. high_value_care_penetration")
print("  4. member_attribution_stability")


GOLD LEVEL 15: METRICS 3 & 4
Gold L15 complete: All 4 metrics calculated

Final Metrics:
  1. risk_adjusted_utilization_rate_final
  2. benefit_design_effectiveness
  3. high_value_care_penetration
  4. member_attribution_stability


# STEP 5: Final Metrics Summary & Save

In [26]:
print("\n" + "="*80)
print("FINAL METRICS SUMMARY")
print("="*80)

final_metrics = gold_l15_final_metrics.select(
    'person_id',
    'payer_concept_id',
    'risk_adjusted_utilization_rate_final',
    'benefit_design_effectiveness',
    'high_value_care_penetration',
    'member_attribution_stability',
    'hcc_risk_score',
    'total_cost_pmpm',
    'enrollment_days'
)

final_metrics.show(10, truncate=False)

agg_stats = final_metrics.agg(
    F.mean('risk_adjusted_utilization_rate_final').alias('avg_util_rate'),
    F.mean('benefit_design_effectiveness').alias('avg_benefit_eff'),
    F.mean('high_value_care_penetration').alias('avg_hv_penetration'),
    F.mean('member_attribution_stability').alias('avg_stability')
).collect()[0]

print("\nAggregate Metrics Across All Members:")
print(f"  Avg Risk-Adjusted Utilization Rate: {agg_stats['avg_util_rate']:.2f}")
print(f"  Avg Benefit Design Effectiveness: {agg_stats['avg_benefit_eff']:.2f}")
print(f"  Avg High-Value Care Penetration: {agg_stats['avg_hv_penetration']:.2f}")
print(f"  Avg Member Attribution Stability: {agg_stats['avg_stability']:.2f}")


FINAL METRICS SUMMARY


+---------+----------------+------------------------------------+----------------------------+---------------------------+----------------------------+--------------+---------------+---------------+
|person_id|payer_concept_id|risk_adjusted_utilization_rate_final|benefit_design_effectiveness|high_value_care_penetration|member_attribution_stability|hcc_risk_score|total_cost_pmpm|enrollment_days|
+---------+----------------+------------------------------------+----------------------------+---------------------------+----------------------------+--------------+---------------+---------------+
|803      |0.0             |0.0                                 |70.0                        |0.0                        |39.0                        |1.0           |0.0            |0              |
|3424     |0.0             |0.0                                 |70.0                        |0.0                        |39.0                        |1.0           |0.0            |0              |
|4590


Aggregate Metrics Across All Members:
  Avg Risk-Adjusted Utilization Rate: 0.01
  Avg Benefit Design Effectiveness: 70.00
  Avg High-Value Care Penetration: 0.00
  Avg Member Attribution Stability: 38.99


In [27]:
metrics_csv = f"{LOCAL_DATA_DIR}/final_payer_metrics.csv"
final_metrics.toPandas().to_csv(metrics_csv, index=False)
print(f"\nSaved final metrics to: {metrics_csv}")


Saved final metrics to: ./5_data/final_payer_metrics.csv


# STEP 6: DAG Construction

In [28]:
# ============================================================================
# STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3) - Payer Utilization Intelligence
# ============================================================================
# Features:
# - withColumn: extracts ALL columns (including chained operations)
# - agg metrics: extracts F.sum, F.count, F.avg, F.countDistinct, etc.
# - join: detects join operations with on/how parameters
# - filter: detects filter operations with conditions
# - select: detects column selections
# - withColumnRenamed: detects column rename operations
# - Window functions: detects row_number, percent_rank, etc.
# - Handles intermediate DataFrames (condition_util, procedure_util, etc.)
# - Supports vertical Gold layer chain (L1 → L2 → ... → L15)
# - 4 Final Metrics integrated into Gold L15
# ============================================================================

print("\n" + "="*80)
print("STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)")
print("="*80)

import json
import re
import networkx as nx
from datetime import datetime
from typing import List, Dict, Tuple

# ============================================================================
# CONFIGURATION - Update these paths as needed
# ============================================================================
NOTEBOOK_FILE = "./5_payer_utilization_intelligence.ipynb"

# Use existing LOCAL_DATA_DIR or set default
try:
    LOCAL_DATA_DIR
except NameError:
    LOCAL_DATA_DIR = "./5_data"
    import os
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# ============================================================================
# NODE DESCRIPTIONS - 5번 노트북 테이블 설명
# ============================================================================
NODE_DESCRIPTIONS = {
    # Bronze Layer (10 tables)
    "bronze_person": "OMOP: Patient demographics and birth information",
    "bronze_obs_period": "OMOP: Observation period records",
    "bronze_payer": "OMOP: Payer plan period information",
    "bronze_condition": "OMOP: Condition occurrence records",
    "bronze_procedure": "OMOP: Procedure occurrence records",
    "bronze_drug": "OMOP: Drug exposure records",
    "bronze_cost": "OMOP: Cost records for healthcare services",
    "bronze_care_site": "OMOP: Healthcare facility information",
    "bronze_provider": "OMOP: Healthcare provider information",
    "bronze_inpatient": "Medicare: Inpatient charges 2011",
    
    # Intermediate DataFrames
    "condition_util": "Intermediate: Condition utilization aggregation by person",
    "procedure_util": "Intermediate: Procedure utilization aggregation by person",
    "drug_util": "Intermediate: Drug utilization aggregation by person",
    "payer_mapping": "Intermediate: Payer period ID to person mapping",
    "member_provider": "Intermediate: Member-provider visit relationship",
    "plan_agg": "Intermediate: Plan-level member aggregation",
    "cost_agg": "Intermediate: Cost aggregation by payer",
    "plan_benchmarks": "Intermediate: Plan-level PMPM benchmarks",
    "member_base": "Intermediate: Base member information selection",
    "plan_cost_share": "Intermediate: Plan cost share percentage selection",
    
    # Silver Layer (6 tables)
    "silver_member_enrollment": "Member enrollment with demographics and payer info",
    "silver_utilization": "Service utilization metrics (conditions, procedures, drugs)",
    "silver_cost_by_payer": "Cost attribution by payer (total paid, payer/patient split)",
    "silver_provider_attribution": "Provider attribution based on visit frequency",
    "silver_risk_scores": "HCC-like risk score calculation (age + condition factors)",
    "silver_plan_features": "Plan benefit design features (member count, cost share)",
    
    # Gold Layer (15 vertical levels)
    "gold_l1_member_profile": "L1: Integrated member profile (enrollment + utilization + cost + provider + risk)",
    "gold_l2_utilization_density": "L2: Utilization density metrics (annual rates)",
    "gold_l3_risk_adjusted": "L3: Risk-adjusted utilization",
    "gold_l4_pmpm": "L4: Cost Per Member Per Month (PMPM)",
    "gold_l5_risk_adjusted_pmpm": "L5: Risk-adjusted PMPM",
    "gold_l6_coverage": "L6: Plan coverage effectiveness (+ silver_plan_features)",
    "gold_l7_high_value": "L7: High-value service utilization",
    "gold_l8_attribution": "L8: Provider attribution continuity",
    "gold_l9_stability": "L9: Member tenure and stability",
    "gold_l10_benchmarked": "L10: Plan performance benchmarking",
    "gold_l11_vbc": "L11: Value-based care alignment",
    "gold_l12_engagement": "L12: Member engagement index",
    "gold_l13_metric1": "L13: METRIC 1 - Risk-Adjusted Utilization Rate",
    "gold_l14_metric2": "L14: METRIC 2 - Benefit Design Effectiveness",
    "gold_l15_final_metrics": "L15: All 4 Final Metrics (includes Metrics 3 & 4)",
    
    # Final Metrics (integrated into Gold L15)
    "final_metrics": "Final output selection of 4 KPIs with key attributes",
}

# ============================================================================
# LINEAGE PARSER CLASS
# ============================================================================
class LineageParserV3:
    """
    Improved PySpark Lineage Parser for Payer Utilization Intelligence Pipeline
    - Parses notebook code cells to extract DataFrame transformations
    - Supports: withColumn, groupBy+agg, join, filter, select, fillna, withColumnRenamed
    - Handles intermediate DataFrames and vertical Gold layer chain
    - Outputs edge-centric lineage format for RAG retrieval
    """
    
    def __init__(self):
        self.edges = []
        self.G = nx.DiGraph()
    
    def get_layer(self, node_id: str) -> str:
        """Extract layer from node ID (bronze/silver/gold/metric)"""
        node_lower = node_id.lower()
        if 'bronze' in node_lower:
            return 'bronze'
        elif 'silver' in node_lower:
            return 'silver'
        elif 'gold' in node_lower:
            return 'gold'
        elif 'final' in node_lower or 'metric' in node_lower:
            return 'metric'
        return 'intermediate'
    
    def extract_all_withcolumns(self, code_block: str) -> List[str]:
        """Extract all column names from withColumn operations"""
        columns = []
        normalized = re.sub(r'\s+', ' ', code_block)
        pattern = r'\.withColumn\s*\(\s*["\']([^"\']+)["\']'
        for match in re.finditer(pattern, normalized):
            col = match.group(1)
            if col not in columns:
                columns.append(col)
        return columns
    
    def extract_column_renames(self, code_block: str) -> List[Dict[str, str]]:
        """Extract column renames from withColumnRenamed operations"""
        renames = []
        normalized = re.sub(r'\s+', ' ', code_block)
        pattern = r'\.withColumnRenamed\s*\(\s*["\']([^"\']+)["\']\s*,\s*["\']([^"\']+)["\']\s*\)'
        for match in re.finditer(pattern, normalized):
            renames.append({"from": match.group(1), "to": match.group(2)})
        return renames
    
    def extract_agg_metrics(self, agg_block: str) -> Dict[str, str]:
        """Extract all metrics from agg block (F.sum, F.count, etc.)"""
        metrics = {}
        normalized = re.sub(r'\s+', ' ', agg_block)
        
        patterns = [
            r'F\s*\.\s*(\w+)\s*\(\s*"([^"]*)"\s*\)\s*\.\s*alias\s*\(\s*"([^"]+)"\s*\)',
            r"F\s*\.\s*(\w+)\s*\(\s*'([^']*)'\s*\)\s*\.\s*alias\s*\(\s*'([^']+)'\s*\)",
            r'F\s*\.\s*(\w+)\s*\(\s*"\*"\s*\)\s*\.\s*alias\s*\(\s*"([^"]+)"\s*\)',
            r"F\s*\.\s*(\w+)\s*\(\s*'\*'\s*\)\s*\.\s*alias\s*\(\s*'([^']+)'\s*\)",
        ]
        
        for pattern in patterns[:2]:
            for match in re.finditer(pattern, normalized):
                func = match.group(1)
                col = match.group(2)
                alias = match.group(3)
                metrics[alias] = f"{func}({col})"
        
        for pattern in patterns[2:]:
            for match in re.finditer(pattern, normalized):
                func = match.group(1)
                alias = match.group(2)
                metrics[alias] = f"{func}(*)"
        
        return metrics
    
    def extract_groupby_cols(self, groupby_str: str) -> List[str]:
        """Extract column names from groupBy clause"""
        cols = []
        for match in re.finditer(r'["\']([^"\']+)["\']', groupby_str):
            cols.append(match.group(1))
        return cols
    
    def extract_filter_condition(self, code_block: str) -> str:
        """Extract filter condition"""
        pattern = r'\.filter\s*\(\s*([^)]+)\s*\)'
        match = re.search(pattern, code_block)
        if match:
            cond = match.group(1).strip()
            cond = re.sub(r'F\.col\s*\(\s*["\']([^"\']+)["\']\s*\)', r'\1', cond)
            return cond[:100]
        return None
    
    def extract_join_info(self, code_block: str) -> List[Dict[str, str]]:
        """Extract ALL join information"""
        joins = []
        normalized = re.sub(r'\s+', ' ', code_block)
        
        # Pattern 1: .join(df.select(...), condition, "type")
        pattern_select = r'\.join\s*\(\s*(\w+)\.select\s*\([^)]+\)\s*,\s*([^,]+)\s*,\s*["\'](\w+)["\']\s*\)'
        for match in re.finditer(pattern_select, normalized):
            other = match.group(1)
            on_clause = match.group(2).strip()
            how = match.group(3)
            
            col_match = re.search(r'["\']([^"\']+)["\']', on_clause)
            on_col = col_match.group(1) if col_match else on_clause[:50]
            
            joins.append({
                "other": other,
                "on": on_col,
                "how": how
            })
        
        # Pattern 2: .join(df, df.col == df2.col, "type")
        pattern_eq = r'\.join\s*\(\s*(\w+)(?:\.select\([^)]*\))?\s*,\s*(\w+\.\w+)\s*==\s*(?:F\.col\(["\'][^"\']+["\']\)|[^,]+)\s*,\s*["\'](\w+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_eq, normalized):
                other = match.group(1)
                on_col = match.group(2)
                how = match.group(3)
                joins.append({"other": other, "on": on_col, "how": how})
        
        # Pattern 3: Simple .join(df, "col", "how")
        pattern_simple = r'\.join\s*\(\s*(\w+)\s*,\s*["\']([^"\']+)["\']\s*,\s*["\'](\w+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_simple, normalized):
                joins.append({
                    "other": match.group(1),
                    "on": match.group(2),
                    "how": match.group(3)
                })
        
        # Pattern 4: Complex condition join
        pattern_complex = r'\.join\s*\(\s*([^,]+)\s*,\s*([^,]+==\s*[^,]+)\s*,\s*["\'](\w+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_complex, normalized):
                other_raw = match.group(1).strip()
                # Extract DataFrame name
                df_match = re.match(r'(\w+)', other_raw)
                other = df_match.group(1) if df_match else other_raw[:20]
                on_col = match.group(2).strip()[:50]
                how = match.group(3)
                joins.append({"other": other, "on": on_col, "how": how})
        
        return joins if joins else None
    
    def extract_select_cols(self, code_block: str) -> List[str]:
        """Extract columns from select operation"""
        cols = []
        # Match .select('col1', 'col2', ...) or .select(F.col('x').alias('y'))
        pattern = r'\.select\s*\(\s*([^)]+)\s*\)'
        match = re.search(pattern, code_block)
        if match:
            select_str = match.group(1)
            for col_match in re.finditer(r'["\']([^"\']+)["\']', select_str):
                col = col_match.group(1)
                if col not in cols:
                    cols.append(col)
        return cols
    
    def extract_fillna_info(self, code_block: str) -> bool:
        """Check if fillna is used"""
        return '.fillna(' in code_block
    
    def extract_coalesce_info(self, code_block: str) -> bool:
        """Check if coalesce is used"""
        return 'F.coalesce(' in code_block
    
    def extract_window_functions(self, code_block: str) -> List[str]:
        """Extract Window function usage"""
        funcs = []
        if 'F.row_number()' in code_block or 'row_number()' in code_block:
            funcs.append('row_number')
        if 'F.lead(' in code_block:
            funcs.append('lead')
        if 'F.lag(' in code_block:
            funcs.append('lag')
        if 'percent_rank()' in code_block:
            funcs.append('percent_rank')
        if 'rank()' in code_block and 'percent_rank' not in code_block:
            funcs.append('rank')
        return funcs
    
    def extract_drop_cols(self, code_block: str) -> List[str]:
        """Extract dropped columns"""
        cols = []
        pattern = r'\.drop\s*\(\s*([^)]+)\s*\)'
        for match in re.finditer(pattern, code_block):
            drop_str = match.group(1)
            for col_match in re.finditer(r'["\']([^"\']+)["\']', drop_str):
                cols.append(col_match.group(1))
        return cols
    
    def parse_assignment(self, code: str) -> List[Dict]:
        """Parse DataFrame assignment statements from code"""
        edges = []
        lines = code.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            assign_match = re.match(r'(\w+)\s*=\s*\(?\s*(\w+)?', line)
            if assign_match and '=' in line and not line.startswith('#'):
                target = assign_match.group(1)
                
                # Skip non-DataFrame variables
                skip = {'spark', 'print', 'if', 'for', 'while', 'def', 'class', 
                       'client', 'query', 'output_file', 'stats', 'rag_data', 
                       'dag_file', 'G', 'lineage_edges', 'edges', 'node_attrs',
                       'label', 'layer', 'in_degree', 'out_degree', 'parents',
                       'children', 'texts', 'w', 'W', 'GCP_PROJECT', 'BQ_CLIENT',
                       'LOCAL_DATA_DIR', 'DATASETS', 'downloaded_files', 'path',
                       'tables_to_download', 'metrics_csv', 'agg_stats', 'csv_path',
                       'full_table'}
                if target.lower() in skip or target in skip:
                    i += 1
                    continue
                
                # Collect full statement
                full_statement = line
                paren_count = line.count('(') - line.count(')')
                backslash_continue = line.rstrip().endswith('\\')
                j = i + 1
                
                while (paren_count > 0 or backslash_continue) and j < len(lines):
                    next_line = lines[j]
                    full_statement += '\n' + next_line
                    paren_count += next_line.count('(') - next_line.count(')')
                    backslash_continue = next_line.rstrip().endswith('\\')
                    j += 1
                
                # Process relevant DataFrames
                keywords = ['bronze', 'silver', 'gold', 'final', 'metric',
                           'member', 'payer', 'condition', 'procedure', 'drug',
                           'cost', 'provider', 'plan', 'risk', 'util', 'enrollment']
                if any(x in full_statement.lower() for x in keywords):
                    edge = self.parse_single_assignment(target, full_statement)
                    if edge:
                        edges.append(edge)
                
                i = j
            else:
                i += 1
        
        return edges
    
    def parse_single_assignment(self, target: str, full_statement: str) -> Dict:
        """Parse a single DataFrame assignment statement"""
        # Find source DataFrame
        source_match = re.search(r'=\s*\(?\s*(\w+)(?:\s*\\)?\s*\.', full_statement)
        if not source_match:
            return None
        
        source = source_match.group(1)
        
        # Skip non-DataFrame sources
        if source.lower() in {'spark', 'f', 'w', 'os', 'pd', 'nx', 'json', 'print', 'datetime', 'bq_client'}:
            return None
        
        sources = [source]
        op_types = []
        
        # Parse joins (multiple joins in gold_l1)
        joins_info = self.extract_join_info(full_statement)
        if joins_info:
            for join in joins_info:
                if join['other'] not in sources:
                    sources.append(join['other'])
            op_types.append('join')
        
        # Parse filter
        filter_cond = self.extract_filter_condition(full_statement)
        if filter_cond:
            op_types.append('filter')
        
        # Parse groupBy
        groupby_match = re.search(r'\.groupBy\s*\(\s*([^)]+)\s*\)', full_statement)
        groupby_cols = []
        if groupby_match:
            groupby_cols = self.extract_groupby_cols(groupby_match.group(1))
            op_types.append('agg')
        
        # Parse agg metrics
        metrics = {}
        agg_start = full_statement.find('.agg(')
        if agg_start != -1:
            paren_count = 0
            content_start = agg_start + 5
            content_end = content_start
            
            for idx, char in enumerate(full_statement[agg_start:]):
                if char == '(':
                    paren_count += 1
                elif char == ')':
                    paren_count -= 1
                    if paren_count == 0:
                        content_end = agg_start + idx
                        break
            
            agg_content = full_statement[content_start:content_end]
            metrics = self.extract_agg_metrics(agg_content)
        
        # Parse withColumn
        columns = self.extract_all_withcolumns(full_statement)
        if columns:
            op_types.append('withColumn')
        
        # Parse withColumnRenamed
        renames = self.extract_column_renames(full_statement)
        if renames:
            op_types.append('rename')
        
        # Parse select
        select_cols = self.extract_select_cols(full_statement)
        if select_cols and not op_types:
            op_types.append('select')
        
        # Parse fillna
        if self.extract_fillna_info(full_statement):
            op_types.append('fillna')
        
        # Parse coalesce
        if self.extract_coalesce_info(full_statement):
            op_types.append('coalesce')
        
        # Parse window functions
        window_funcs = self.extract_window_functions(full_statement)
        if window_funcs:
            op_types.append('window')
        
        # Parse drop
        drop_cols = self.extract_drop_cols(full_statement)
        if drop_cols:
            op_types.append('drop')
        
        # Skip if no operations found
        if not op_types:
            return None
        
        op_str = '+'.join(op_types)
        
        # Build operation dict
        operation = {"op": op_str}
        if joins_info:
            operation["joins"] = joins_info
        if filter_cond:
            operation["filter"] = filter_cond
        if groupby_cols:
            operation["groupBy"] = groupby_cols
        if metrics:
            operation["metrics"] = metrics
        if columns:
            operation["columns"] = columns
        if renames:
            operation["renames"] = renames
        if select_cols:
            operation["select"] = select_cols
        if window_funcs:
            operation["window_functions"] = window_funcs
        if drop_cols:
            operation["drop"] = drop_cols
        
        # Generate description text
        text_parts = [f"{target} is created from {source}:"]
        if joins_info:
            for j in joins_info:
                text_parts.append(f"{j['how']}-joins with {j['other']} on {j['on']}")
        if filter_cond:
            text_parts.append(f"filters by {filter_cond[:50]}")
        if groupby_cols:
            text_parts.append(f"groups by {groupby_cols}")
        if metrics:
            text_parts.append(f"computes {list(metrics.keys())}")
        if columns:
            text_parts.append(f"adds columns {columns}")
        if renames:
            text_parts.append(f"renames {[r['from'] + '->' + r['to'] for r in renames]}")
        if window_funcs:
            text_parts.append(f"uses window functions {window_funcs}")
        if drop_cols:
            text_parts.append(f"drops {drop_cols}")
        
        text = " ".join(text_parts) + "."
        
        return {
            "id": f"{' + '.join(sources)} -> {target} ({op_str})",
            "source_nodes": sources,
            "target_node": target,
            "operation": operation,
            "text": text
        }
    
    def parse_notebook(self, notebook_path: str) -> Tuple[List[Dict], nx.DiGraph]:
        """Parse entire notebook file and extract lineage"""
        print(f"\n📖 Reading notebook: {notebook_path}")
        
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
        
        code_cells = []
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code':
                source = cell.get('source', [])
                code = ''.join(source) if isinstance(source, list) else source
                code_cells.append(code)
        
        print(f"   Found {len(code_cells)} code cells")
        
        all_edges = []
        for code in code_cells:
            edges = self.parse_assignment(code)
            all_edges.extend(edges)
        
        # Deduplicate by target (keep most complete)
        seen_targets = {}
        for edge in all_edges:
            target = edge['target_node']
            if target not in seen_targets:
                seen_targets[target] = edge
            else:
                existing = seen_targets[target]
                existing_score = len(existing.get('operation', {}).get('columns', [])) + \
                                len(existing.get('operation', {}).get('metrics', {})) + \
                                len(existing['source_nodes'])
                new_score = len(edge.get('operation', {}).get('columns', [])) + \
                           len(edge.get('operation', {}).get('metrics', {})) + \
                           len(edge['source_nodes'])
                if new_score > existing_score:
                    seen_targets[target] = edge
        
        self.edges = list(seen_targets.values())
        self.build_graph()
        
        print(f"   Extracted {len(self.edges)} transformation edges")
        
        return self.edges, self.G
    
    def build_graph(self):
        """Build NetworkX DAG from edges"""
        self.G = nx.DiGraph()
        
        for edge in self.edges:
            target = edge['target_node']
            if not self.G.has_node(target):
                self.G.add_node(target, id=target, label=target, layer=self.get_layer(target))
            
            for src in edge['source_nodes']:
                if not self.G.has_node(src):
                    self.G.add_node(src, id=src, label=src, layer=self.get_layer(src))
                self.G.add_edge(src, target, etype="consume")
    
    def generate_rag_data(self) -> List[Dict]:
        """Generate RAG-compatible node data"""
        rag_data = []
        
        for node_id in self.G.nodes():
            layer = self.get_layer(node_id)
            in_deg = self.G.in_degree(node_id)
            out_deg = self.G.out_degree(node_id)
            parents = list(self.G.predecessors(node_id))
            children = list(self.G.successors(node_id))
            
            description = NODE_DESCRIPTIONS.get(node_id, node_id)
            
            texts = [
                description,
                f"Layer: {layer}",
                f"Incoming edges: {in_deg}, Outgoing edges: {out_deg}"
            ]
            if parents:
                texts.append(f"Consumes: {', '.join(parents)}")
            if children:
                texts.append(f"Feeds into: {', '.join(children)}")
            
            rag_data.append({"id": node_id, "texts": texts})
        
        return rag_data


# ============================================================================
# MAIN EXECUTION
# ============================================================================
print("\n" + "="*60)
print("Parsing Notebook for PySpark Transformations")
print("="*60)

parser = LineageParserV3()

try:
    edges, G = parser.parse_notebook(NOTEBOOK_FILE)
    rag_data = parser.generate_rag_data()
    
    # Statistics
    bronze_count = len([n for n in G.nodes() if parser.get_layer(n) == 'bronze'])
    silver_count = len([n for n in G.nodes() if parser.get_layer(n) == 'silver'])
    gold_count = len([n for n in G.nodes() if parser.get_layer(n) == 'gold'])
    metric_count = len([n for n in G.nodes() if parser.get_layer(n) == 'metric'])
    intermediate_count = len([n for n in G.nodes() if parser.get_layer(n) == 'intermediate'])
    
    print("\n" + "="*60)
    print("Parsing Results")
    print("="*60)
    print(f"  Total nodes: {G.number_of_nodes()}")
    print(f"  Total graph edges: {G.number_of_edges()}")
    print(f"  Lineage records: {len(edges)}")
    print(f"  Bronze: {bronze_count}, Silver: {silver_count}, Gold: {gold_count}, Metric: {metric_count}")
    print(f"  Intermediate: {intermediate_count}")
    print(f"  Is DAG: {nx.is_directed_acyclic_graph(G)}")
    
    # ========================================================================
    # SAVE OUTPUTS
    # ========================================================================
    print("\n" + "="*60)
    print("Saving Outputs")
    print("="*60)
    
    lineage_file = f"{LOCAL_DATA_DIR}/payer_utilization_lineage_edges_auto.json"
    with open(lineage_file, 'w', encoding='utf-8') as f:
        json.dump(edges, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved lineage: {lineage_file}")
    
    rag_file = f"{LOCAL_DATA_DIR}/payer_utilization_rag_data_auto.json"
    with open(rag_file, 'w', encoding='utf-8') as f:
        json.dump(rag_data, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved RAG data: {rag_file}")
    
    dag_file = f"{LOCAL_DATA_DIR}/payer_utilization_dag_auto.graphml"
    nx.write_graphml(G, dag_file)
    print(f"✔ Saved DAG: {dag_file}")
    
    stats = {
        "pipeline": "Payer Utilization Intelligence (Auto-Parsed v3)",
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "lineage_edges": len(edges),
        "bronze_nodes": bronze_count,
        "silver_nodes": silver_count,
        "gold_nodes": gold_count,
        "metric_nodes": metric_count,
        "intermediate_nodes": intermediate_count,
        "is_dag": nx.is_directed_acyclic_graph(G),
        "timestamp": datetime.now().isoformat(),
        "source_notebook": NOTEBOOK_FILE
    }
    
    stats_file = f"{LOCAL_DATA_DIR}/dag_statistics_auto.json"
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved statistics: {stats_file}")
    
    # ========================================================================
    # DISPLAY RESULTS
    # ========================================================================
    print("\n" + "="*80)
    print("AUTO-PARSING COMPLETE (v3)")
    print("="*80)
    
    print("\n--- Extracted Transformations ---\n")
    for i, edge in enumerate(edges, 1):
        print(f"{i}. [{edge['operation']['op']}] {edge['id']}")
        print(f"   Sources: {edge['source_nodes']}")
        print(f"   Target:  {edge['target_node']}")
        
        op = edge['operation']
        if 'columns' in op and op['columns']:
            print(f"   Columns: {op['columns']}")
        if 'groupBy' in op and op['groupBy']:
            print(f"   GroupBy: {op['groupBy']}")
        if 'metrics' in op and op['metrics']:
            print(f"   Metrics: {op['metrics']}")
        if 'joins' in op:
            for j in op['joins']:
                print(f"   Join:    {j['how']} on '{j['on']}' with {j['other']}")
        if 'filter' in op:
            print(f"   Filter:  {op['filter']}")
        if 'window_functions' in op:
            print(f"   Window:  {op['window_functions']}")
        if 'renames' in op:
            print(f"   Renames: {op['renames']}")
        if 'drop' in op:
            print(f"   Drop:    {op['drop']}")
        print(f"   Text:    {edge['text']}")
        print()
    
    print("="*80)
    print(f"Total: {len(edges)} transformations, {G.number_of_nodes()} nodes")
    print(f"Files saved to: {LOCAL_DATA_DIR}/")
    print("="*80)

except FileNotFoundError:
    print(f"\n❌ Error: Notebook file not found: {NOTEBOOK_FILE}")
    print("   Update NOTEBOOK_FILE path at the top of this cell.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()


STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)

Parsing Notebook for PySpark Transformations

📖 Reading notebook: ./5_payer_utilization_intelligence.ipynb
   Found 38 code cells
   Extracted 32 transformation edges

Parsing Results
  Total nodes: 40
  Total graph edges: 46
  Lineage records: 32
  Bronze: 8, Silver: 6, Gold: 15, Metric: 1
  Intermediate: 10
  Is DAG: True

Saving Outputs
✔ Saved lineage: ./5_data/payer_utilization_lineage_edges_auto.json
✔ Saved RAG data: ./5_data/payer_utilization_rag_data_auto.json
✔ Saved DAG: ./5_data/payer_utilization_dag_auto.graphml
✔ Saved statistics: ./5_data/dag_statistics_auto.json

AUTO-PARSING COMPLETE (v3)

--- Extracted Transformations ---

1. [join+withColumn+rename+coalesce+drop] bronze_person + bronze_obs_period + bronze_payer -> silver_member_enrollment (join+withColumn+rename+coalesce+drop)
   Sources: ['bronze_person', 'bronze_obs_period', 'bronze_payer']
   Target:  silver_member_enrollment
   Columns: ['birth_datetime_convert